### Cell 06.01 — freeze the SCN region and its evidence

In [ ]:
# Cell 06.01
# Freeze the suggestive SCN FI3 region for literature
# comparison and subsequent candidate-gene analysis.

from pathlib import Path
import pandas as pd
import numpy as np


def find_project_root():
    """Locate the repository root from the current working directory."""
    current = Path.cwd().resolve()
    for path in (current, *current.parents):
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    raise FileNotFoundError(
        "Repository root not found. Run this notebook from within "
        "the flyer-hartwig-qtl-reanalysis repository."
    )

PROJECT_ROOT = find_project_root()


QTL_PHYSICAL_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_qtl_physical_localization.xlsx"
)

QTL_SCREEN_FILE = (
    PROJECT_ROOT
    / "results"
    / "qtl"
    / "flyer_hartwig_all33_empirical_qtl_screen.xlsx"
)


physical_regions = pd.read_excel(
    QTL_PHYSICAL_FILE,
    sheet_name="candidate_regions"
)

all33 = pd.read_excel(
    QTL_SCREEN_FILE,
    sheet_name="all_33_empirical"
)

scn_region_profile = pd.read_excel(
    QTL_SCREEN_FILE,
    sheet_name="region_scn_fi3"
)


scn_region = (
    physical_regions
    .loc[
        physical_regions["trait"]
        == "scn_fi3"
    ]
    .iloc[0]
)


scn_peak = (
    all33
    .loc[
        all33["trait"]
        == "scn_fi3"
    ]
    .iloc[0]
)


print("FROZEN SCN FI3 REGION")
print("=" * 100)

print(
    f"Chromosome: {scn_region['chromosome']}"
)

print(
    f"Physical bracket: "
    f"{scn_region['region_start_mb']:.3f}–"
    f"{scn_region['region_end_mb']:.3f} Mb"
)

print(
    f"Span: "
    f"{scn_region['region_span_mb']:.3f} Mb"
)

print(
    f"Peak marker: "
    f"{scn_peak['peak_marker']}"
)

print(
    f"LOD: "
    f"{scn_peak['lod']:.3f}"
)

print(
    f"Genome-wide empirical P: "
    f"{scn_peak['peak_empirical_p']:.5f}"
)

print(
    f"Status: "
    f"{scn_peak['genomewide_status']}"
)

### Cell 06.02 — construct a literature-comparison table
* For now, enter only literature regions for which we have defensible chromosome/physical information. This table separates direct source evidence from our FxH region.

In [ ]:
# Cell 06.02
# Literature comparison table for SCN-associated Gm20 regions.
#
# Coordinates from different genome assemblies/publications
# must NOT be treated as directly equivalent without checking
# the underlying assembly.

literature_scn_gm20 = pd.DataFrame(
    [
        {
            "study_label":
                "Flyer_x_Hartwig_current_reanalysis",

            "resistance_source":
                "Flyer/Hartwig",

            "chromosome":
                "Gm20",

            "start_mb":
                scn_region["region_start_mb"],

            "end_mb":
                scn_region["region_end_mb"],

            "peak_or_candidate":
                "Satt354",

            "trait_or_HG_type":
                "SCN FI3",

            "assembly_or_coordinate_basis":
                "Wm82.gnm6 SoySSR anchors",

            "evidence":
                "suggestive_10pct",

            "notes":
                (
                    "Anchor-defined physical bracket; "
                    "not a statistical confidence interval"
                )
        },

        {
            "study_label":
                "PI437655_SCN_QTL",

            "resistance_source":
                "PI 437655",

            "chromosome":
                "Gm20",

            "start_mb":
                np.nan,

            "end_mb":
                np.nan,

            "peak_or_candidate":
                "Chr20 QTL",

            "trait_or_HG_type":
                (
                    "HG 1.3.5.6.7 and "
                    "HG 1.2.3.4.5.6.7"
                ),

            "assembly_or_coordinate_basis":
                "publication-specific",

            "evidence":
                "published_QTL",

            "notes":
                (
                    "Independent evidence for SCN "
                    "resistance on chromosome 20"
                )
        },

        {
            "study_label":
                "PI507354_Chr20_QTL",

            "resistance_source":
                "PI 507354",

            "chromosome":
                "Gm20",

            "start_mb":
                42.81,

            "end_mb":
                43.82,

            "peak_or_candidate":
                "Glyma.20G197600 candidate",

            "trait_or_HG_type":
                "SCN resistance",

            "assembly_or_coordinate_basis":
                "publication-specific",

            "evidence":
                "published_QTL",

            "notes":
                (
                    "Recent Chr20 SCN QTL; coordinates "
                    "must be assembly-checked before "
                    "formal overlap claims"
                )
        }
    ]
)


display(
    literature_scn_gm20
)

### Cell 06.03 — calculate provisional physical overlap/gap only where coordinates exist
* This is descriptive only until assembly compatibility is verified.

In [ ]:
# Cell 06.03
# Compare published coordinate ranges with the current
# FxH anchor-defined region.
#
# CAUTION:
# These comparisons are provisional unless the published
# coordinates are confirmed to use the same/reference-compatible
# Wm82 assembly.

fxh_start = float(
    scn_region["region_start_mb"]
)

fxh_end = float(
    scn_region["region_end_mb"]
)


comparison_rows = []


for _, row in literature_scn_gm20.iterrows():

    if row["study_label"] == (
        "Flyer_x_Hartwig_current_reanalysis"
    ):
        continue

    if (
        pd.notna(row["start_mb"])
        and pd.notna(row["end_mb"])
    ):

        lit_start = float(
            row["start_mb"]
        )

        lit_end = float(
            row["end_mb"]
        )


        overlap_start = max(
            fxh_start,
            lit_start
        )

        overlap_end = min(
            fxh_end,
            lit_end
        )

        overlap_mb = max(
            0.0,
            overlap_end - overlap_start
        )


        if overlap_mb > 0:

            relationship = (
                "coordinate_overlap"
            )

            gap_mb = 0.0

        else:

            relationship = (
                "no_coordinate_overlap"
            )

            gap_mb = max(
                lit_start - fxh_end,
                fxh_start - lit_end,
                0.0
            )


        comparison_rows.append({
            "study_label":
                row["study_label"],

            "literature_start_mb":
                lit_start,

            "literature_end_mb":
                lit_end,

            "fxh_start_mb":
                fxh_start,

            "fxh_end_mb":
                fxh_end,

            "overlap_mb":
                overlap_mb,

            "gap_mb":
                gap_mb,

            "relationship":
                relationship,

            "assembly_check_required":
                True
        })


scn_gm20_coordinate_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


display(
    scn_gm20_coordinate_comparison
)

### Cell 06.04 — flag an especially relevant historical Gm20 observation
* There is another feature worth tracking separately: Satt354 itself has historical QTL associations on LG I/Gm20, including SDS. SoyBase lists Satt354 at 46.22 cM on the GmComposite2003_I map, and reviews identify an SDS-associated region at Satt354 on chromosome 20.

In [ ]:
# Cell 06.04
# Record marker-level historical context separately
# from SCN-QTL overlap claims.

marker_context = pd.DataFrame(
    [
        {
            "marker": "Satt354",
            "historical_linkage_group": "I",
            "modern_chromosome": "Gm20",
            "historical_context":
                (
                    "Marker has previously been associated "
                    "with multiple QTL, including SDS-related "
                    "variation on chromosome 20."
                ),
            "interpretation":
                (
                    "Relevant marker-level historical context, "
                    "but not evidence that the present SCN "
                    "signal is the same causal locus."
                )
        }
    ]
)


display(
    marker_context
)

### Cell 06.05 — correct the literature metadata

In [ ]:
# Cell 06.05
# Correct literature assembly metadata before any
# physical-overlap interpretation.

literature_scn_gm20.loc[
    literature_scn_gm20[
        "study_label"
    ] == "PI507354_Chr20_QTL",
    "assembly_or_coordinate_basis"
] = "Wm82.a2.v1"


literature_scn_gm20.loc[
    literature_scn_gm20[
        "study_label"
    ] == "PI507354_Chr20_QTL",
    "notes"
] = (
    "Chr20 QTL reported at ~42.81–43.82 Mb "
    "on Wm82.a2.v1. Coordinates cannot be "
    "directly compared with FxH Wm82.gnm6 "
    "coordinates without assembly translation."
)


literature_scn_gm20.loc[
    literature_scn_gm20[
        "study_label"
    ] == "PI437655_SCN_QTL",
    "assembly_or_coordinate_basis"
] = (
    "genetic linkage map / BARC SNP markers"
)


display(
    literature_scn_gm20
)

### Cell 06.06 — invalidate cross-assembly overlap calculations
* Rather than deleting Cell 06.03, keep it as provenance and make the validity explicit.

In [ ]:
# Cell 06.06
# Flag whether coordinate overlap is actually interpretable.

scn_gm20_coordinate_comparison[
    "fxh_assembly"
] = "Wm82.gnm6"

scn_gm20_coordinate_comparison[
    "literature_assembly"
] = "Wm82.a2.v1"


scn_gm20_coordinate_comparison[
    "same_assembly"
] = (
    scn_gm20_coordinate_comparison[
        "fxh_assembly"
    ]
    ==
    scn_gm20_coordinate_comparison[
        "literature_assembly"
    ]
)


scn_gm20_coordinate_comparison[
    "coordinate_relationship_valid"
] = (
    scn_gm20_coordinate_comparison[
        "same_assembly"
    ]
)


# Do not retain a biological overlap/gap interpretation
# when assemblies differ.
scn_gm20_coordinate_comparison[
    "harmonized_relationship"
] = np.where(
    scn_gm20_coordinate_comparison[
        "same_assembly"
    ],
    scn_gm20_coordinate_comparison[
        "relationship"
    ],
    "not_comparable_until_assembly_translation"
)


scn_gm20_coordinate_comparison[
    "harmonized_gap_mb"
] = np.where(
    scn_gm20_coordinate_comparison[
        "same_assembly"
    ],
    scn_gm20_coordinate_comparison[
        "gap_mb"
    ],
    np.nan
)


display(
    scn_gm20_coordinate_comparison[
        [
            "study_label",
            "literature_start_mb",
            "literature_end_mb",
            "literature_assembly",
            "fxh_start_mb",
            "fxh_end_mb",
            "fxh_assembly",
            "same_assembly",
            "harmonized_relationship",
            "harmonized_gap_mb"
        ]
    ]
)

### Cell 06.07 — check what harmonization resources are already local

In [ ]:
# Cell 06.07
# Check locally available reference resources before
# downloading or generating anything new.

REFERENCE_DIR = (
    PROJECT_ROOT
    / "reference"
    / "soybase"
)


print("REFERENCE DIRECTORY")
print(REFERENCE_DIR)
print()


if REFERENCE_DIR.exists():

    reference_files = sorted(
        [
            p.name
            for p in REFERENCE_DIR.iterdir()
            if p.is_file()
        ]
    )

    print(
        f"FILES FOUND: {len(reference_files)}"
    )

    for name in reference_files:
        print(name)

else:

    print(
        "Reference directory does not exist."
    )

### Cell 06.08 — search local filenames for useful resources

In [ ]:
# Cell 06.08
# Search existing SoyBase reference files for resources
# useful for literature-QTL harmonization.

search_terms = [
    "snp",
    "soySNP",
    "6K",
    "50K",
    "ann",
    "gene",
    "pangene",
    "correspond",
    "gnm6"
]


matching_files = []


for path in REFERENCE_DIR.rglob("*"):

    if not path.is_file():
        continue

    name_lower = path.name.lower()

    matched_terms = [
        term
        for term in search_terms
        if term.lower() in name_lower
    ]

    if matched_terms:

        matching_files.append({
            "file": path.name,
            "relative_path":
                str(
                    path.relative_to(
                        PROJECT_ROOT
                    )
                ),
            "matched_terms":
                ", ".join(
                    matched_terms
                ),
            "size_kb":
                round(
                    path.stat().st_size
                    / 1024,
                    2
                )
        })


local_reference_inventory = (
    pd.DataFrame(
        matching_files
    )
)


print(
    "POTENTIALLY USEFUL LOCAL "
    "HARMONIZATION RESOURCES"
)
print("=" * 120)


if len(
    local_reference_inventory
):

    display(
        local_reference_inventory
        .sort_values(
            "file"
        )
    )

else:

    print(
        "No additional matching files found."
    )

### Cell 06.09 — prepare local reference paths
* First create a clean place for the gnm6 annotation.

In [ ]:
# Cell 06.09
# Prepare local paths for Wm82.gnm6 gene annotation.

SOYBASE_REF_DIR = (
    PROJECT_ROOT
    / "reference"
    / "soybase"
)

SOYBASE_REF_DIR.mkdir(
    parents=True,
    exist_ok=True
)


GNM6_GENE_GFF = (
    SOYBASE_REF_DIR
    / "glyma.Wm82.gnm6.ann1.PKSW.gene_models_main.gff3.gz"
)


print("REFERENCE DIRECTORY:")
print(SOYBASE_REF_DIR)

print("\nEXPECTED GNM6 GENE FILE:")
print(GNM6_GENE_GFF)

print("\nALREADY EXISTS:")
print(GNM6_GENE_GFF.exists())

### Cell 06.10 — verify and inspect the downloaded annotation
* Run this after the download.

In [ ]:
# Cell 06.10
# Verify the Wm82.gnm6 annotation and inspect GFF3 structure.

import gzip


print(
    "GNM6 GENE ANNOTATION EXISTS:",
    GNM6_GENE_GFF.exists()
)


if GNM6_GENE_GFF.exists():

    print(
        "FILE SIZE MB:",
        round(
            GNM6_GENE_GFF.stat().st_size
            / 1024**2,
            2
        )
    )

    print("\nFIRST NON-COMMENT RECORDS")
    print("=" * 120)

    n_shown = 0

    with gzip.open(
        GNM6_GENE_GFF,
        "rt"
    ) as fh:

        for line in fh:

            if line.startswith("#"):
                continue

            print(
                line.rstrip()[:250]
            )

            n_shown += 1

            if n_shown >= 5:
                break

### Cell 06.11 — parse only gene features from Wm82.gnm6

In [ ]:
# Cell 06.11
# Parse Wm82.gnm6 gene features into a compact DataFrame.

def parse_gff_attributes(text):

    attrs = {}

    for item in str(text).split(";"):

        if "=" not in item:
            continue

        key, value = item.split(
            "=",
            1
        )

        attrs[key] = value

    return attrs


gff_columns = [
    "seqid",
    "source",
    "feature_type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes"
]


gnm6_gff = pd.read_csv(
    GNM6_GENE_GFF,
    sep="\t",
    comment="#",
    names=gff_columns,
    dtype={
        "seqid": str,
        "feature_type": str,
        "attributes": str
    },
    compression="gzip"
)


gnm6_genes = (
    gnm6_gff
    .loc[
        gnm6_gff[
            "feature_type"
        ] == "gene"
    ]
    .copy()
)


gnm6_genes[
    "attribute_dict"
] = (
    gnm6_genes[
        "attributes"
    ]
    .map(
        parse_gff_attributes
    )
)


# Capture commonly used ID fields without assuming
# every record uses exactly the same attributes.
for field in [
    "ID",
    "Name",
    "gene_id"
]:

    gnm6_genes[field] = (
        gnm6_genes[
            "attribute_dict"
        ]
        .map(
            lambda d:
                d.get(
                    field,
                    np.nan
                )
        )
    )


print(
    "TOTAL GNM6 GENE FEATURES:",
    len(gnm6_genes)
)

print(
    "CHROMOSOMES/SEQIDS:",
    gnm6_genes[
        "seqid"
    ].nunique()
)


display(
    gnm6_genes[
        [
            "seqid",
            "start",
            "end",
            "strand",
            "ID",
            "Name",
            "gene_id"
        ]
    ]
    .head(10)
)

### Cell 06.12 — identify the actual gnm6 chromosome-20 sequence name
* Do this rather than assuming the GFF uses exactly "Gm20".

In [ ]:
# Cell 06.12
# Inspect sequence identifiers containing chromosome 20.

seqids = sorted(
    gnm6_genes[
        "seqid"
    ]
    .dropna()
    .unique()
)


chr20_candidates = [
    seqid
    for seqid in seqids
    if (
        "20" in str(seqid)
        or str(seqid).lower()
        in [
            "gm20",
            "chr20",
            "20"
        ]
    )
]


print("POSSIBLE CHROMOSOME-20 SEQIDS")
print("=" * 100)

for x in chr20_candidates:
    print(x)


print("\nALL SEQIDS")
print("=" * 100)

print(seqids)

### Cell 06.13 — extract all Wm82.gnm6 genes in the FxH SCN interval

In [ ]:
# Cell 06.13
# Extract all Wm82.gnm6 gene models overlapping the
# frozen FxH SCN FI3 physical bracket.

SCN_CHR_SEQID = "glyma.Wm82.gnm6.Gm20"

SCN_START_BP = int(
    scn_region["region_start_bp"]
)

SCN_END_BP = int(
    scn_region["region_end_bp"]
)


scn_gm20_genes = (
    gnm6_genes
    .loc[
        (
            gnm6_genes["seqid"]
            == SCN_CHR_SEQID
        )
        &
        (
            gnm6_genes["end"]
            >= SCN_START_BP
        )
        &
        (
            gnm6_genes["start"]
            <= SCN_END_BP
        )
    ]
    .copy()
    .sort_values(
        ["start", "end"]
    )
    .reset_index(drop=True)
)


scn_gm20_genes[
    "start_mb"
] = (
    scn_gm20_genes["start"]
    / 1e6
)

scn_gm20_genes[
    "end_mb"
] = (
    scn_gm20_genes["end"]
    / 1e6
)


print("FXH SCN FI3 GENE INVENTORY")
print("=" * 100)

print(
    f"Region: Gm20 "
    f"{SCN_START_BP:,}–"
    f"{SCN_END_BP:,} bp"
)

print(
    f"Genes overlapping region: "
    f"{len(scn_gm20_genes):,}"
)


display(
    scn_gm20_genes[
        [
            "Name",
            "start",
            "end",
            "start_mb",
            "end_mb",
            "strand",
            "ID"
        ]
    ]
    .head(20)
)

### Cell 06.14 — inspect annotation fields available for these genes
* The GFF clearly contains more information than ID and Name, so let's see which attributes are actually available before deciding how to annotate candidates.

In [ ]:
# Cell 06.14
# Inspect GFF3 attribute keys available among genes
# in the FxH SCN interval.

attribute_keys = sorted(
    {
        key
        for d in scn_gm20_genes[
            "attribute_dict"
        ]
        for key in d.keys()
    }
)


print(
    "ATTRIBUTE KEYS PRESENT IN "
    "SCN-REGION GENE RECORDS"
)
print("=" * 100)

for key in attribute_keys:
    print(key)

### Cell 06.15 — search the full gnm6 annotation for Glyma.20G197600
* This is important because the gnm6 GFF already contains ancestry/correspondence metadata. We may be able to translate the Wm82.a2.v1 gene without another file.

In [ ]:
# Cell 06.15
# Search all Wm82.gnm6 gene attributes for the historical
# candidate gene name Glyma.20G197600.

OLD_CANDIDATE = "Glyma.20G197600"


candidate_name_hits = (
    gnm6_genes
    .loc[
        gnm6_genes[
            "attributes"
        ]
        .str.contains(
            OLD_CANDIDATE,
            case=False,
            na=False,
            regex=False
        )
    ]
    .copy()
)


print(
    f"GNM6 GENE RECORDS CONTAINING "
    f"{OLD_CANDIDATE}: "
    f"{len(candidate_name_hits)}"
)

display(
    candidate_name_hits[
        [
            "seqid",
            "start",
            "end",
            "strand",
            "ID",
            "Name",
            "attributes"
        ]
    ]
)

### Cell 06.16 — inspect ancestorIdentifier and candidate-related annotation

In [ ]:
# Cell 06.16
# Extract ancestry metadata for Gm20 genes and inspect
# whether historical gene IDs are preserved.

gnm6_genes[
    "ancestorIdentifier"
] = (
    gnm6_genes[
        "attribute_dict"
    ]
    .map(
        lambda d:
            d.get(
                "ancestorIdentifier",
                np.nan
            )
    )
)


gm20_genes = (
    gnm6_genes
    .loc[
        gnm6_genes["seqid"]
        == SCN_CHR_SEQID
    ]
    .copy()
)


candidate_ancestor_hits = (
    gm20_genes
    .loc[
        gm20_genes[
            "ancestorIdentifier"
        ]
        .astype(str)
        .str.contains(
            OLD_CANDIDATE,
            case=False,
            na=False,
            regex=False
        )
    ]
    .copy()
)


print(
    "ANCESTOR-IDENTIFIER MATCHES FOR "
    f"{OLD_CANDIDATE}: "
    f"{len(candidate_ancestor_hits)}"
)


display(
    candidate_ancestor_hits[
        [
            "Name",
            "start",
            "end",
            "strand",
            "ancestorIdentifier"
        ]
    ]
)

### Cell 06.17 — quantify the position of Glyma.20G197600 relative to the FxH bracket

In [ ]:
# Cell 06.17
# Quantify the Wm82.gnm6 position of the published
# Glyma.20G197600 candidate relative to the FxH SCN region.

published_candidate_gnm6 = (
    candidate_name_hits
    .iloc[0]
)


candidate_start_bp = int(
    published_candidate_gnm6["start"]
)

candidate_end_bp = int(
    published_candidate_gnm6["end"]
)


if candidate_end_bp < SCN_START_BP:

    positional_relationship = (
        "outside_before_fxh_region"
    )

    distance_to_region_bp = (
        SCN_START_BP
        - candidate_end_bp
    )

elif candidate_start_bp > SCN_END_BP:

    positional_relationship = (
        "outside_after_fxh_region"
    )

    distance_to_region_bp = (
        candidate_start_bp
        - SCN_END_BP
    )

else:

    positional_relationship = (
        "overlaps_fxh_region"
    )

    distance_to_region_bp = 0


published_candidate_comparison = (
    pd.DataFrame(
        [
            {
                "gene":
                    published_candidate_gnm6[
                        "Name"
                    ],

                "gnm6_start_bp":
                    candidate_start_bp,

                "gnm6_end_bp":
                    candidate_end_bp,

                "gnm6_start_mb":
                    candidate_start_bp
                    / 1e6,

                "gnm6_end_mb":
                    candidate_end_bp
                    / 1e6,

                "fxh_region_start_bp":
                    SCN_START_BP,

                "fxh_region_end_bp":
                    SCN_END_BP,

                "relationship":
                    positional_relationship,

                "distance_to_fxh_region_bp":
                    distance_to_region_bp,

                "distance_to_fxh_region_mb":
                    distance_to_region_bp
                    / 1e6,

                "ancestorIdentifier":
                    published_candidate_gnm6[
                        "attribute_dict"
                    ].get(
                        "ancestorIdentifier",
                        np.nan
                    )
            }
        ]
    )
)


display(
    published_candidate_comparison
)

### Cell 06.18 — extract functional annotation fields for the 307 FxH-region genes

In [ ]:
# Cell 06.18
# Extract the annotation fields already embedded in the
# official Wm82.gnm6 GFF3 for the 307 SCN-region genes.

for field in [
    "Note",
    "Ontology_term",
    "Dbxref",
    "ancestorIdentifier"
]:

    scn_gm20_genes[field] = (
        scn_gm20_genes[
            "attribute_dict"
        ]
        .map(
            lambda d, f=field:
                d.get(
                    f,
                    np.nan
                )
        )
    )


scn_gene_inventory = (
    scn_gm20_genes[
        [
            "Name",
            "start",
            "end",
            "start_mb",
            "end_mb",
            "strand",
            "Note",
            "Ontology_term",
            "Dbxref",
            "ancestorIdentifier"
        ]
    ]
    .copy()
)


print(
    "ANNOTATED GENES IN FXH SCN REGION:",
    len(scn_gene_inventory)
)

print(
    "Genes with Note:",
    scn_gene_inventory[
        "Note"
    ].notna().sum()
)

print(
    "Genes with Ontology_term:",
    scn_gene_inventory[
        "Ontology_term"
    ].notna().sum()
)

print(
    "Genes with Dbxref:",
    scn_gene_inventory[
        "Dbxref"
    ].notna().sum()
)


display(
    scn_gene_inventory.head(20)
)

### Cell 06.19 — exploratory defense-related annotation screen
* This is only a candidate-prioritization screen, not evidence that these genes cause the SCN phenotype.

In [ ]:
# Cell 06.19
# Exploratory annotation-based screen for genes with
# functions plausibly relevant to plant defense or
# nematode responses.
#
# This is hypothesis generation only.

def combined_annotation_text(row):

    fields = [
        row.get("Note"),
        row.get("Ontology_term"),
        row.get("Dbxref")
    ]

    return " ".join(
        str(x)
        for x in fields
        if pd.notna(x)
    ).lower()


scn_gene_inventory[
    "annotation_text"
] = (
    scn_gene_inventory
    .apply(
        combined_annotation_text,
        axis=1
    )
)


candidate_keywords = {
    "immune_receptor":
        [
            "nb-arc",
            "nbs-lrr",
            "leucine-rich repeat",
            "disease resistance",
            "resistance protein"
        ],

    "kinase_signaling":
        [
            "kinase",
            "receptor-like",
            "map kinase"
        ],

    "transcription_regulation":
        [
            "transcription factor",
            "wrky",
            "myb",
            "bzip",
            "nac"
        ],

    "hormone_defense":
        [
            "jasmon",
            "salicy",
            "ethylene",
            "auxin"
        ],

    "redox_detox":
        [
            "glutathione",
            "peroxidase",
            "oxidoreductase",
            "reactive oxygen"
        ],

    "cell_wall":
        [
            "cell wall",
            "pectin",
            "cellulose",
            "lignin"
        ]
}


def assign_candidate_categories(text):

    hits = []

    for category, keywords in (
        candidate_keywords.items()
    ):

        if any(
            keyword in text
            for keyword in keywords
        ):

            hits.append(
                category
            )

    return ";".join(
        hits
    )


scn_gene_inventory[
    "candidate_categories"
] = (
    scn_gene_inventory[
        "annotation_text"
    ]
    .map(
        assign_candidate_categories
    )
)


scn_candidate_screen = (
    scn_gene_inventory
    .loc[
        scn_gene_inventory[
            "candidate_categories"
        ] != ""
    ]
    .copy()
    .sort_values(
        [
            "start",
            "Name"
        ]
    )
)


print(
    "GENES FLAGGED BY EXPLORATORY "
    "DEFENSE-RELATED ANNOTATION SCREEN:",
    len(scn_candidate_screen)
)


display(
    scn_candidate_screen[
        [
            "Name",
            "start_mb",
            "end_mb",
            "strand",
            "candidate_categories",
            "Note",
            "Ontology_term"
        ]
    ]
)

### Cell 06.20 — save the SCN physical/literature comparison checkpoint

In [ ]:
# Cell 06.20
# Save the current SCN Gm20 physical localization,
# literature-candidate comparison, and gene inventory.

SCN_CANDIDATE_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_candidate_region.xlsx"
)

SCN_CANDIDATE_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)


with pd.ExcelWriter(
    SCN_CANDIDATE_FILE,
    engine="openpyxl"
) as writer:

    pd.DataFrame(
        [scn_region]
    ).to_excel(
        writer,
        sheet_name="fxh_scn_region",
        index=False
    )

    published_candidate_comparison.to_excel(
        writer,
        sheet_name="published_candidate",
        index=False
    )

    scn_gene_inventory.drop(
        columns=[
            "annotation_text"
        ],
        errors="ignore"
    ).to_excel(
        writer,
        sheet_name="all_307_genes",
        index=False
    )

    scn_candidate_screen.drop(
        columns=[
            "annotation_text"
        ],
        errors="ignore"
    ).to_excel(
        writer,
        sheet_name="defense_gene_screen",
        index=False
    )

    literature_scn_gm20.to_excel(
        writer,
        sheet_name="literature_context",
        index=False
    )


print(
    "Saved:"
)

print(
    SCN_CANDIDATE_FILE
)

### Cell 06.21 — decode annotation text and apply stricter functional rules

In [ ]:
# Cell 06.21
# Build a more conservative, auditable candidate screen
# using primarily the gene Note field.
#
# This replaces the broad substring screen for ranking purposes.

from urllib.parse import unquote


scn_gene_inventory[
    "Note_decoded"
] = (
    scn_gene_inventory["Note"]
    .fillna("")
    .map(unquote)
)


strict_candidate_rules = {

    "receptor_or_defense_signaling": [
        "receptor-like kinase",
        "receptor-like protein kinase",
        "cysteine-rich receptor-like",
        "leucine-rich repeat",
        "disease resistance-responsive",
        "disease resistance protein"
    ],

    "protein_kinase": [
        "protein kinase",
        "serine/threonine kinase"
    ],

    "redox_or_detoxification": [
        "glutathione s-transferase",
        "thioredoxin",
        "peroxidase",
        "oxidoreductase"
    ],

    "cell_wall_or_root_interface": [
        "pectate lyase",
        "cell wall structural protein",
        "cell wall protein",
        "expansin"
    ],

    "defense_related_transcription": [
        "wrky transcription factor",
        "nac transcription factor",
        "myb transcription factor",
        "bhlh",
        "myc-type"
    ]
}


def strict_categories(note):

    text = str(note).lower()

    categories = []

    for category, terms in (
        strict_candidate_rules.items()
    ):

        if any(
            term in text
            for term in terms
        ):
            categories.append(category)

    return ";".join(categories)


scn_gene_inventory[
    "strict_candidate_categories"
] = (
    scn_gene_inventory[
        "Note_decoded"
    ]
    .map(strict_categories)
)


strict_candidate_screen = (
    scn_gene_inventory
    .loc[
        scn_gene_inventory[
            "strict_candidate_categories"
        ] != ""
    ]
    .copy()
    .sort_values(
        ["start", "Name"]
    )
    .reset_index(drop=True)
)


print(
    "STRICT FUNCTIONAL SCREEN:",
    len(strict_candidate_screen),
    "of",
    len(scn_gene_inventory),
    "genes"
)


display(
    strict_candidate_screen[
        [
            "Name",
            "start_mb",
            "end_mb",
            "strand",
            "strict_candidate_categories",
            "Note_decoded"
        ]
    ]
)

### Cell 06.22 — separate particularly interpretable functional classes
* Rather than assigning an arbitrary numerical score, this creates transparent evidence flags.

In [ ]:
# Cell 06.22
# Create transparent functional evidence flags.
# These are prioritization features, not causal evidence.

priority = strict_candidate_screen.copy()

priority["receptor_defense_flag"] = (
    priority[
        "strict_candidate_categories"
    ]
    .str.contains(
        "receptor_or_defense_signaling",
        na=False
    )
)

priority["kinase_flag"] = (
    priority[
        "strict_candidate_categories"
    ]
    .str.contains(
        "protein_kinase",
        na=False
    )
)

priority["redox_flag"] = (
    priority[
        "strict_candidate_categories"
    ]
    .str.contains(
        "redox_or_detoxification",
        na=False
    )
)

priority["cell_wall_flag"] = (
    priority[
        "strict_candidate_categories"
    ]
    .str.contains(
        "cell_wall_or_root_interface",
        na=False
    )
)

priority["transcription_flag"] = (
    priority[
        "strict_candidate_categories"
    ]
    .str.contains(
        "defense_related_transcription",
        na=False
    )
)


priority["n_functional_classes"] = (
    priority[
        [
            "receptor_defense_flag",
            "kinase_flag",
            "redox_flag",
            "cell_wall_flag",
            "transcription_flag"
        ]
    ]
    .sum(axis=1)
)


priority = (
    priority
    .sort_values(
        [
            "n_functional_classes",
            "start_mb"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


display(
    priority[
        [
            "Name",
            "start_mb",
            "n_functional_classes",
            "receptor_defense_flag",
            "kinase_flag",
            "redox_flag",
            "cell_wall_flag",
            "transcription_flag",
            "Note_decoded"
        ]
    ]
)

### Cell 06.23 — examine spatial clustering of the candidates
* One striking feature in your current output is that many interesting annotations occur toward approximately 37.5–38.5 Mb, near the distal end of the FxH physical bracket. Let's quantify rather than eyeball that pattern.

In [ ]:
# Cell 06.23
# Summarize candidate density across 1-Mb windows.
# This is descriptive only.

priority[
    "midpoint_mb"
] = (
    priority["start_mb"]
    + priority["end_mb"]
) / 2


priority[
    "one_mb_bin"
] = (
    np.floor(
        priority["midpoint_mb"]
    )
    .astype(int)
)


candidate_density = (
    priority
    .groupby(
        "one_mb_bin",
        as_index=False
    )
    .agg(
        n_strict_candidates=(
            "Name",
            "count"
        ),
        genes=(
            "Name",
            lambda x:
                "; ".join(x)
        )
    )
)


candidate_density[
    "window"
] = (
    candidate_density[
        "one_mb_bin"
    ].astype(str)
    + "–"
    + (
        candidate_density[
            "one_mb_bin"
        ] + 1
    ).astype(str)
    + " Mb"
)


display(
    candidate_density[
        [
            "window",
            "n_strict_candidates",
            "genes"
        ]
    ]
)

### Cell 06.24 — save the refined screen

In [ ]:
# Cell 06.24
# Add refined functional screens to the existing workbook.

REFINED_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_refined_candidates.xlsx"
)


with pd.ExcelWriter(
    REFINED_FILE,
    engine="openpyxl"
) as writer:

    scn_gene_inventory.drop(
        columns=["annotation_text"],
        errors="ignore"
    ).to_excel(
        writer,
        sheet_name="all_307_genes",
        index=False
    )

    strict_candidate_screen.to_excel(
        writer,
        sheet_name="strict_functional_screen",
        index=False
    )

    priority.to_excel(
        writer,
        sheet_name="functional_evidence",
        index=False
    )

    candidate_density.to_excel(
        writer,
        sheet_name="candidate_density",
        index=False
    )

    published_candidate_comparison.to_excel(
        writer,
        sheet_name="published_candidate",
        index=False
    )


print("Saved:")
print(REFINED_FILE)

### Cell 06.25 — build a title-based candidate screen

In [ ]:
# Cell 06.25
# Use only the primary annotation title before the first
# semicolon for candidate classification.
#
# InterPro/domain text is retained as secondary evidence
# but does not drive the primary classification.

def primary_annotation_title(note):

    if pd.isna(note):
        return ""

    return (
        str(note)
        .split(";")[0]
        .strip()
    )


scn_gene_inventory[
    "primary_annotation"
] = (
    scn_gene_inventory[
        "Note_decoded"
    ]
    .map(
        primary_annotation_title
    )
)


title_rules = {

    "receptor_or_defense_signaling": [
        "receptor-like kinase",
        "receptor-like protein kinase",
        "cysteine-rich receptor-like",
        "disease resistance-responsive",
        "disease resistance protein"
    ],

    "protein_kinase": [
        "protein kinase",
        "serine/threonine kinase"
    ],

    "redox_or_detoxification": [
        "glutathione s-transferase",
        "thioredoxin",
        "peroxidase",
        "oxidoreductase"
    ],

    "cell_wall_or_root_interface": [
        "pectate lyase",
        "cell wall structural protein",
        "cell wall protein",
        "expansin"
    ],

    "defense_related_transcription": [
        "wrky transcription factor",
        "nac transcription factor",
        "myb transcription factor",
        "transcription factor bhlh",
        "myc-type"
    ]
}


def classify_primary_annotation(text):

    text = str(text).lower()

    hits = []

    for category, terms in (
        title_rules.items()
    ):

        if any(
            term in text
            for term in terms
        ):
            hits.append(category)

    return ";".join(hits)


scn_gene_inventory[
    "title_based_categories"
] = (
    scn_gene_inventory[
        "primary_annotation"
    ]
    .map(
        classify_primary_annotation
    )
)


title_candidate_screen = (
    scn_gene_inventory
    .loc[
        scn_gene_inventory[
            "title_based_categories"
        ] != ""
    ]
    .copy()
    .sort_values(
        ["start", "Name"]
    )
    .reset_index(drop=True)
)


print(
    "TITLE-BASED FUNCTIONAL SCREEN:",
    len(title_candidate_screen),
    "of",
    len(scn_gene_inventory),
    "genes"
)


display(
    title_candidate_screen[
        [
            "Name",
            "start_mb",
            "end_mb",
            "strand",
            "title_based_categories",
            "primary_annotation"
        ]
    ]
)

### Cell 06.26 — flag the strongest mechanistic classes
* For SCN, I would not rank every kinase or transcription factor equally.
* This separates genes whose annotation is more directly consistent with host–pathogen recognition, defense signaling, redox response, or infection-site remodeling.

In [ ]:
# Cell 06.26
# Assign transparent mechanistic priority classes.
#
# These are hypothesis-generation tiers, not causal claims.

candidate_priority = (
    title_candidate_screen
    .copy()
)


def assign_priority_tier(row):

    cats = str(
        row[
            "title_based_categories"
        ]
    )

    title = str(
        row[
            "primary_annotation"
        ]
    ).lower()


    # Tier 1:
    # direct defense/receptor annotations
    # or GST/thioredoxin defense-related redox machinery

    if (
        "receptor_or_defense_signaling"
        in cats
    ):

        return "Tier_1"


    if (
        "glutathione s-transferase"
        in title
        or "thioredoxin"
        in title
    ):

        return "Tier_1"


    # Tier 2:
    # signaling, cell-wall, or regulatory candidates

    if (
        "protein_kinase"
        in cats
        or
        "cell_wall_or_root_interface"
        in cats
        or
        "defense_related_transcription"
        in cats
    ):

        return "Tier_2"


    # Other plausible redox functions

    if (
        "redox_or_detoxification"
        in cats
    ):

        return "Tier_3"


    return "Tier_3"


candidate_priority[
    "priority_tier"
] = (
    candidate_priority
    .apply(
        assign_priority_tier,
        axis=1
    )
)


candidate_priority = (
    candidate_priority
    .sort_values(
        [
            "priority_tier",
            "start_mb"
        ]
    )
    .reset_index(drop=True)
)


display(
    candidate_priority[
        [
            "Name",
            "start_mb",
            "priority_tier",
            "title_based_categories",
            "primary_annotation"
        ]
    ]
)

### Cell 06.27 — add literature-evidence columns without overstating them
* Let's distinguish actual soybean evidence from generic annotation.

In [ ]:
# Cell 06.27
# Add literature/context evidence for genes for which
# we currently have specific soybean evidence.
#
# SCN-specific evidence remains False unless directly shown.

candidate_priority[
    "soybean_biotic_stress_evidence"
] = False

candidate_priority[
    "soybean_other_stress_evidence"
] = False

candidate_priority[
    "direct_SCN_evidence"
] = False

candidate_priority[
    "literature_context"
] = ""


# Glyma.20G100500:
# reported in soybean red-leaf-blotch candidate interval
# and in soybean-virus miRNA work.

mask = (
    candidate_priority["Name"]
    == "Glyma.20G100500"
)

candidate_priority.loc[
    mask,
    "soybean_biotic_stress_evidence"
] = True

candidate_priority.loc[
    mask,
    "literature_context"
] = (
    "Reported as an LRR-family candidate in a "
    "soybean red-leaf-blotch GWAS region and as "
    "a predicted miRNA target in soybean mosaic "
    "virus work; not direct SCN evidence."
)


# Glyma.20G080700:
# reported in soybean salinity-response GWAS.

mask = (
    candidate_priority["Name"]
    == "Glyma.20G080700"
)

candidate_priority.loc[
    mask,
    "soybean_other_stress_evidence"
] = True

candidate_priority.loc[
    mask,
    "literature_context"
] = (
    "Receptor-like protein kinase associated "
    "with soybean salinity-response variation; "
    "not direct SCN evidence."
)


display(
    candidate_priority[
        [
            "Name",
            "priority_tier",
            "soybean_biotic_stress_evidence",
            "soybean_other_stress_evidence",
            "direct_SCN_evidence",
            "literature_context"
        ]
    ]
    .loc[
        (
            candidate_priority[
                "soybean_biotic_stress_evidence"
            ]
        )
        |
        (
            candidate_priority[
                "soybean_other_stress_evidence"
            ]
        )
    ]
)

### Cell 06.28 — save the final candidate-prioritization checkpoint

In [ ]:
# Cell 06.28
# Save title-based and literature-aware candidate tables.

FINAL_CANDIDATE_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_candidate_prioritization.xlsx"
)


with pd.ExcelWriter(
    FINAL_CANDIDATE_FILE,
    engine="openpyxl"
) as writer:

    scn_gene_inventory.to_excel(
        writer,
        sheet_name="all_307_genes",
        index=False
    )

    title_candidate_screen.to_excel(
        writer,
        sheet_name="title_based_screen",
        index=False
    )

    candidate_priority.to_excel(
        writer,
        sheet_name="candidate_priority",
        index=False
    )

    published_candidate_comparison.to_excel(
        writer,
        sheet_name="published_candidate",
        index=False
    )


print("Saved:")
print(FINAL_CANDIDATE_FILE)

### Cell 06.29 — upgrade the evidence table using direct SCN literature

In [ ]:
# Cell 06.29
# Add peer-reviewed SCN-specific molecular evidence.
#
# Important:
# "direct_SCN_evidence" here means the gene itself has been
# reported in an SCN infection/syncytium molecular dataset.
# It does NOT mean causal resistance has been demonstrated.

candidate_priority[
    "direct_SCN_evidence"
] = False

candidate_priority[
    "scn_evidence_type"
] = ""

candidate_priority[
    "scn_evidence_reference"
] = ""

candidate_priority[
    "scn_evidence_interpretation"
] = ""


mask = (
    candidate_priority["Name"]
    == "Glyma.20G104000"
)


candidate_priority.loc[
    mask,
    "direct_SCN_evidence"
] = True


candidate_priority.loc[
    mask,
    "scn_evidence_type"
] = (
    "SCN-associated differential methylation "
    "+ syncytium differential expression"
)


candidate_priority.loc[
    mask,
    "scn_evidence_reference"
] = (
    "Rambani et al. 2015, Plant Physiology "
    "168:1364-1377; DOI 10.1104/pp.15.00826"
)


candidate_priority.loc[
    mask,
    "scn_evidence_interpretation"
] = (
    "Gene reported among hypomethylated genes "
    "overlapping SCN-induced syncytium DEGs; "
    "supports involvement in soybean-SCN response "
    "but does not demonstrate causal resistance."
)


display(
    candidate_priority.loc[
        candidate_priority[
            "direct_SCN_evidence"
        ],
        [
            "Name",
            "start_mb",
            "priority_tier",
            "primary_annotation",
            "direct_SCN_evidence",
            "scn_evidence_type",
            "scn_evidence_reference",
            "scn_evidence_interpretation"
        ]
    ]
)

### Cell 06.30 — create evidence-based candidate tiers

In [ ]:
# Cell 06.30
# Replace the original purely functional Tier 1 label with
# evidence-aware candidate classes.

def assign_evidence_class(row):

    if row["direct_SCN_evidence"]:
        return "Tier_1A_direct_SCN_molecular_evidence"

    if row["priority_tier"] == "Tier_1":
        return "Tier_1B_strong_functional_candidate"

    if row["priority_tier"] == "Tier_2":
        return "Tier_2_plausible_functional_candidate"

    return "Tier_3_broad_functional_candidate"


candidate_priority[
    "evidence_class"
] = (
    candidate_priority
    .apply(
        assign_evidence_class,
        axis=1
    )
)


evidence_rank = {
    "Tier_1A_direct_SCN_molecular_evidence": 1,
    "Tier_1B_strong_functional_candidate": 2,
    "Tier_2_plausible_functional_candidate": 3,
    "Tier_3_broad_functional_candidate": 4
}


candidate_priority[
    "evidence_rank"
] = (
    candidate_priority[
        "evidence_class"
    ]
    .map(evidence_rank)
)


candidate_priority = (
    candidate_priority
    .sort_values(
        [
            "evidence_rank",
            "start_mb"
        ]
    )
    .reset_index(drop=True)
)


display(
    candidate_priority[
        [
            "Name",
            "start_mb",
            "evidence_class",
            "primary_annotation",
            "direct_SCN_evidence",
            "soybean_biotic_stress_evidence",
            "soybean_other_stress_evidence"
        ]
    ]
)

### Cell 06.31 — summarize only the strongest candidates

In [ ]:
# Cell 06.31
# Generate a compact manuscript-oriented shortlist.

scn_shortlist = (
    candidate_priority
    .loc[
        candidate_priority[
            "evidence_rank"
        ] <= 2
    ]
    .copy()
)


scn_shortlist[
    "distance_from_distal_boundary_mb"
] = (
    SCN_END_BP / 1e6
    - scn_shortlist[
        "start_mb"
    ]
)


print(
    "HIGH-PRIORITY SCN CANDIDATE SHORTLIST:",
    len(scn_shortlist)
)


display(
    scn_shortlist[
        [
            "Name",
            "start_mb",
            "evidence_class",
            "primary_annotation",
            "direct_SCN_evidence",
            "soybean_biotic_stress_evidence",
            "distance_from_distal_boundary_mb"
        ]
    ]
)

### Cell 06.32 — save the literature-integrated checkpoint

In [ ]:
# Cell 06.32
# Save literature-integrated SCN candidate prioritization.

LITERATURE_INTEGRATED_FILE = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_literature_integrated_candidates.xlsx"
)


with pd.ExcelWriter(
    LITERATURE_INTEGRATED_FILE,
    engine="openpyxl"
) as writer:

    scn_gene_inventory.to_excel(
        writer,
        sheet_name="all_307_genes",
        index=False
    )

    title_candidate_screen.to_excel(
        writer,
        sheet_name="title_based_24",
        index=False
    )

    candidate_priority.to_excel(
        writer,
        sheet_name="evidence_ranked",
        index=False
    )

    scn_shortlist.to_excel(
        writer,
        sheet_name="high_priority_shortlist",
        index=False
    )

    published_candidate_comparison.to_excel(
        writer,
        sheet_name="published_G197600",
        index=False
    )


print("Saved:")
print(LITERATURE_INTEGRATED_FILE)

### Cell 06.33 — create the final manuscript-oriented nine-gene table

In [ ]:
# Cell 06.33
# Final manuscript-oriented shortlist for the FxH SCN Gm20 region.

scn_manuscript_candidates = (
    scn_shortlist[
        [
            "Name",
            "start_mb",
            "end_mb",
            "strand",
            "evidence_class",
            "primary_annotation",
            "direct_SCN_evidence",
            "soybean_biotic_stress_evidence",
            "soybean_other_stress_evidence",
            "distance_from_distal_boundary_mb"
        ]
    ]
    .copy()
)


scn_manuscript_candidates = (
    scn_manuscript_candidates
    .rename(
        columns={
            "Name": "gene",
            "start_mb": "start_mb_gnm6",
            "end_mb": "end_mb_gnm6",
            "primary_annotation": "functional_annotation",
            "distance_from_distal_boundary_mb":
                "distance_to_fxh_distal_boundary_mb"
        }
    )
)


scn_manuscript_candidates[
    "interpretation"
] = np.where(
    scn_manuscript_candidates[
        "direct_SCN_evidence"
    ],
    (
        "Prior SCN-responsive molecular evidence; "
        "candidate for follow-up, not validated "
        "as the causal FxH resistance gene."
    ),
    (
        "Functionally plausible candidate within "
        "the FxH physical bracket; no direct "
        "SCN-specific evidence established here."
    )
)


display(
    scn_manuscript_candidates
)

### Cell 06.34 — formally summarize the distal candidate cluster

In [ ]:
# Cell 06.34
# Descriptive characterization of the candidate-rich distal region.
# This is NOT a refined QTL confidence interval.

DISTAL_CUTOFF_MB = 37.4

distal_candidates = (
    scn_manuscript_candidates
    .loc[
        scn_manuscript_candidates[
            "start_mb_gnm6"
        ] >= DISTAL_CUTOFF_MB
    ]
    .copy()
)


print("DISTAL CANDIDATE CLUSTER")
print("=" * 90)

print(
    "Descriptive cutoff:",
    f">= {DISTAL_CUTOFF_MB:.1f} Mb"
)

print(
    "High-priority genes in distal cluster:",
    len(distal_candidates)
)

print(
    "Fraction of high-priority shortlist:",
    f"{len(distal_candidates) / len(scn_manuscript_candidates):.1%}"
)

print(
    "Cluster span:",
    f"{distal_candidates['start_mb_gnm6'].min():.3f}"
    "–"
    f"{distal_candidates['end_mb_gnm6'].max():.3f} Mb"
)

print(
    "\nIMPORTANT: This is descriptive candidate clustering, "
    "not a statistically refined QTL interval."
)


display(
    distal_candidates[
        [
            "gene",
            "start_mb_gnm6",
            "functional_annotation",
            "evidence_class",
            "direct_SCN_evidence"
        ]
    ]
)

### Cell 06.35 — make a final SCN-region evidence summary

In [ ]:
# Cell 06.35
# Compact summary of the complete FxH SCN Gm20 analysis.

scn_region_final_summary = pd.DataFrame(
    [
        {
            "trait":
                "scn_fi3",

            "peak_marker":
                "Satt354",

            "structural_group":
                "pLG01b_Gm20",

            "physical_chr":
                "Gm20",

            "lod":
                2.713316,

            "empirical_p":
                0.068693,

            "significance_status":
                "suggestive_10pct",

            "effect_2_minus_0":
                -28.889,

            "favorable_parent":
                "Flyer",

            "region_start_bp_gnm6":
                SCN_START_BP,

            "region_end_bp_gnm6":
                SCN_END_BP,

            "region_span_mb":
                (
                    SCN_END_BP
                    - SCN_START_BP
                ) / 1e6,

            "region_definition":
                "anchor_defined_physical_bracket",

            "genes_in_region":
                len(scn_gene_inventory),

            "title_based_candidates":
                len(title_candidate_screen),

            "high_priority_candidates":
                len(scn_manuscript_candidates),

            "direct_scn_evidence_candidates":
                int(
                    scn_manuscript_candidates[
                        "direct_SCN_evidence"
                    ].sum()
                ),

            "top_evidence_gene":
                "Glyma.20G104000",

            "published_Glyma20G197600_overlap":
                False,

            "Glyma20G197600_distance_mb":
                float(
                    published_candidate_comparison.loc[
                        0,
                        "distance_to_fxh_region_mb"
                    ]
                )
        }
    ]
)


display(
    scn_region_final_summary.T
)

### Cell 06.36 — final Notebook 06 export

In [ ]:
# Cell 06.36
# Final frozen export for Notebook 06.

NOTEBOOK06_FINAL = (
    PROJECT_ROOT
    / "results"
    / "candidate_genes"
    / "flyer_hartwig_scn_gm20_final_evidence_summary.xlsx"
)


with pd.ExcelWriter(
    NOTEBOOK06_FINAL,
    engine="openpyxl"
) as writer:

    scn_region_final_summary.to_excel(
        writer,
        sheet_name="region_summary",
        index=False
    )

    scn_manuscript_candidates.to_excel(
        writer,
        sheet_name="high_priority_9",
        index=False
    )

    distal_candidates.to_excel(
        writer,
        sheet_name="distal_cluster",
        index=False
    )

    candidate_priority.to_excel(
        writer,
        sheet_name="all_ranked_24",
        index=False
    )

    scn_gene_inventory.to_excel(
        writer,
        sheet_name="all_region_307",
        index=False
    )

    published_candidate_comparison.to_excel(
        writer,
        sheet_name="published_G197600",
        index=False
    )


print("FINAL NOTEBOOK 06 EXPORT")
print("=" * 90)
print(NOTEBOOK06_FINAL)
print()
print("Notebook 06 can now be frozen.")